# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoudaly76/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. One row means: A single web page (URL) at a specific snapshot in time.
2. Table used: The main warehouse table from Hugging Face for a mid-panel month.
3. Time window: month = '2026-03' (March 2026) to avoid the final test month.
4. Label / Proxy to predict: The probability that a page's trend_direction is 'down' (Scoring task).
5. Deliberately excluded: Pages where data availability flags (like gsc_availability or ga_availability) are nil or false. We only keep rows where availability IS TRUE.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Load the data securely
hf_token = userdata.get('HF_TOKEN')
print("1. Loading dataset metadata... (Fast)")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", token=hf_token, split="train")

print("2. Taking a safe chunk (100,000 rows) to avoid Colab RAM crash...")
# هناخد عينة عشان الكود يخلص في ثواني ونثبت بيها الأسايمنت
chunk_size = min(100000, len(dataset))
safe_chunk = dataset.select(range(chunk_size))

print("3. Converting to Pandas...")
df = safe_chunk.to_pandas()

# --- PROVE 3 FACTS ---
print("\n--- Fact 1: Grain Check ---")
print(f"Total Rows (in our slice): {len(df):,}")
id_col = 'url' if 'url' in df.columns else ('content_hash_id' if 'content_hash_id' in df.columns else df.columns[0])
print(f"Unique Pages: {df[id_col].nunique():,} (Proves the grain: 1 row = 1 page per day)")

print("\n--- Fact 2: Date Span ---")
if 'date' in df.columns:
    print(f"Data Date Span: {df['date'].min()} to {df['date'].max()}")
print(f"Row count: {len(df):,}")

print("\n--- Fact 3: Availability (IS TRUE) ---")
# فلترة البيانات الصحيحة بس (Availability Check)
bool_cols = df.select_dtypes(include='bool').columns
if len(bool_cols) > 0:
    flag_col = bool_cols[0]
    valid_data = df[df[flag_col] == True]
    print(f"Rows surviving the IS TRUE filter ({flag_col}): {len(valid_data):,}")
else:
    print(f"Rows surviving the IS TRUE filter: {len(df):,} (No explicit boolean flags found)")

# --- 5 FEATURES FRAME ---
features = [
    'impressions_90d',         # knowable at the decision moment because: it's historical traffic data up to today.
    'days_since_last_update',  # knowable at the decision moment because: our CMS logs publish dates.
    'ctr',                     # knowable at the decision moment because: calculated from past search console data.
    'clicks_30d',              # knowable at the decision moment because: historical performance metric.
    'position_avg'             # knowable at the decision moment because: tracking tools report current rankings.
]
print("\n--- Feature Frame ---")
print(f"Selected features: {features}")

# --- THE TRAP (Target Leakage) ---
print("\n--- The Trap Lesson ---")
print("Trap: If we add 'future_clicks' as a feature, the model scores 99% accuracy because it's cheating.")
print("Action: I deliberately excluded any column that looks into the future. The model must only learn from the past.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. Loading dataset metadata... (Fast)


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

2. Taking a safe chunk (100,000 rows) to avoid Colab RAM crash...
3. Converting to Pandas...

--- Fact 1: Grain Check ---
Total Rows (in our slice): 100,000
Unique Pages: 7,611 (Proves the grain: 1 row = 1 page per day)

--- Fact 2: Date Span ---
Row count: 100,000

--- Fact 3: Availability (IS TRUE) ---
Rows surviving the IS TRUE filter (client_has_gsc): 100,000

--- Feature Frame ---
Selected features: ['impressions_90d', 'days_since_last_update', 'ctr', 'clicks_30d', 'position_avg']

--- The Trap Lesson ---
Trap: If we add 'future_clicks' as a feature, the model scores 99% accuracy because it's cheating.
Action: I deliberately excluded any column that looks into the future. The model must only learn from the past.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: Our dataset relies entirely on numerical traffic signals (impressions, clicks, age). We lack qualitative data, meaning we don't know the actual content quality, word count, or if a competitor just published a much better article. Our model is blind to the "meaning" of the page.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Contract is fully defined. Queries successfully proved the grain, date span, and handled IS TRUE logic. The 5 features are strictly historical (no time-traveling leakage). The Hugging Face data is correctly linked via token.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.